In [1]:
import os
import getpass

if not os.environ.get("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass.getpass("Digite sua GROQ_API_KEY: ")
print("✅ GROQ_API_KEY configurada!")

✅ GROQ_API_KEY configurada!


In [ ]:
from typing import List
from dataclasses import dataclass
from pydantic import BaseModel, Field
from pydantic_ai import Agent, RunContext
from dotenv import load_dotenv

# Carregar variáveis de ambiente
load_dotenv()

# Definir a estrutura de dependências para recuperação
@dataclass
class RetrievalDependencies:
    documents: List[str]

# Definir o modelo de resultado esperado
class RAGResult(BaseModel):
    answer: str = Field(..., description="Resposta final gerada com base no contexto recuperado")
    retrieved_docs: List[str] = Field(..., description="Lista de documentos relevantes utilizados na resposta")

# Criar o agente RAG
rag_agent = Agent(
    'groq:openai/gpt-oss-120b',  # Substituído por modelo Groq que você já usa
    deps_type=RetrievalDependencies,
    output_type=RAGResult,
    system_prompt="Você é um assistente que utiliza informações recuperadas para responder com precisão. Use os documentos relevantes para compor sua resposta."
)

# Definir a ferramenta de recuperação de contexto
@rag_agent.tool()  # Adicionei parênteses aqui
async def retrieve_context(ctx: RunContext[RetrievalDependencies], query: str) -> List[str]:
    """
    Recupera documentos relevantes com base na consulta.
    
    Args:
        ctx: Contexto contendo os documentos disponíveis
        query: Consulta para filtrar documentos relevantes
        
    Returns:
        Lista de documentos que correspondem à consulta
    """
    # Busca simplificada: filtra documentos que contenham a query
    relevant = [doc for doc in ctx.deps.documents if query.lower() in doc.lower()]
    return relevant if relevant else ["Nenhum documento relevante encontrado."]

# Função principal adaptada para Jupyter
async def executar_rag():
    # Criar as dependências com documentos de exemplo
    dependencies = RetrievalDependencies(
        documents=[
            "PydanticAI: framework voltado à criação de agentes de IA, com foco em validação de dados, respostas estruturadas, execução de ferramentas e injeção de dependências.",
            "LangGraph: framework voltado à orquestração de agentes e fluxos de trabalho, permitindo gerenciar estados, controlar etapas de execução, implementar decisões condicionais e ciclos.",
            "O PydanticAI e o LangGraph são frameworks Python utilizados no desenvolvimento de aplicações e agentes de inteligência artificial baseados em modelos de linguagem (LLMs)."
        ]
    )
    
    # Executar o agente
    result = await rag_agent.run("Explique como o usa O LangGraph, juntamente com o PydanticAI.", deps=dependencies)
    
    print("Exemplo RAG:")
    print(f"Resposta: {result.output.answer}")
    print("\nDocumentos recuperados:")
    for i, doc in enumerate(result.output.retrieved_docs):
        print(f"{i+1}. {doc}")
    
    return result

# Em Jupyter, basta chamar com await - sem asyncio.run()
await executar_rag()

Exemplo RAG:
Resposta: O LangGraph é um framework de orquestração de agentes e fluxos de trabalho que permite definir e controlar o ciclo de vida de um agente, gerenciar seu estado interno, inserir decisões condicionais e loops. O PydanticAI, por sua vez, fornece modelos de dados tipados e validação automática (baseados no Pydantic) para simplificar a construção de prompts, respostas e parâmetros usados pelos LLMs.

### Como combinar os dois
1. **Definir a estrutura de dados com PydanticAI**
   - Crie classes que herdam de `BaseModel` (ou das classes específicas do PydanticAI) para representar entradas, saídas e configurações do seu agente. Por exemplo:
     ```python
     from pydantic import BaseModel

     class Pedido(BaseModel):
         cliente: str
         produto: str
         quantidade: int
     ```
   - Essas classes garantem que os dados que entram e saem do fluxo do LangGraph estejam sempre validados e tipados.

2. **Construir nós (nodes) do LangGraph**
   - Cada nó pode 

AgentRunResult(output=RAGResult(answer='O LangGraph é um framework de orquestração de agentes e fluxos de trabalho que permite definir e controlar o ciclo de vida de um agente, gerenciar seu estado interno, inserir decisões condicionais e loops. O PydanticAI, por sua vez, fornece modelos de dados tipados e validação automática (baseados no Pydantic) para simplificar a construção de prompts, respostas e parâmetros usados pelos LLMs.\n\n### Como combinar os dois\n1. **Definir a estrutura de dados com PydanticAI**\n   - Crie classes que herdam de `BaseModel` (ou das classes específicas do PydanticAI) para representar entradas, saídas e configurações do seu agente. Por exemplo:\n     ```python\n     from pydantic import BaseModel\n\n     class Pedido(BaseModel):\n         cliente: str\n         produto: str\n         quantidade: int\n     ```\n   - Essas classes garantem que os dados que entram e saem do fluxo do LangGraph estejam sempre validados e tipados.\n\n2. **Construir nós (nodes) d